In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 3

In [2]:
import sys
!{sys.executable} -m pip install faiss-cpu
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.metrics import average_precision_score

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))
    
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

# Loading Pipeline Models
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1) # set device=0 if GPU is available
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")

# Helper Data for Row 150
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150_text = str(row_150[row_150['answer']])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.2 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Question -1

In [3]:
res_q1 = zs(prompt_150, candidate_labels=labels_150)
q1_prob = res_q1['scores'][res_q1['labels'].index(ans_150_text)]
print(f"Q1 Answer: {q1_prob:.3f}")

Q1 Answer: 0.384


### Question - 2

In [4]:
prompt_150_embed = model.encode([prompt_150], show_progress_bar=False)
D, I = index.search(prompt_150_embed, k=10)
retrieved_indices_150 = I[0].tolist()

if 150 in retrieved_indices_150:
    q2_rank = retrieved_indices_150.index(150) + 1
else:
    q2_rank = -1
print(f"Q2 Answer: Rank {q2_rank}")

Q2 Answer: Rank 10


### Question - 3 

In [5]:
docs_10 = [kb[i] for i in retrieved_indices_150]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

ranked_docs_with_src_idx = sorted(zip(retrieved_indices_150, ce_scores), key=lambda x: x[1], reverse=True)
reranked_indices = [x[0] for x in ranked_docs_with_src_idx]

if 150 in reranked_indices:
    q3_rank = reranked_indices.index(150) + 1
else:
    q3_rank = -1
print(f"Q3 Answer: Rank {q3_rank}")

Q3 Answer: Rank 1


### Question - 4 

In [6]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
prompt_42_embed = model.encode([prompt_42], show_progress_bar=False)
_, I_42 = index.search(prompt_42_embed, k=5)

docs_5_q4 = [kb[i] for i in I_42[0]]
concatenated_docs = " ".join(docs_5_q4)
rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"

tokens_42 = tokenizer_bert.tokenize(rag_string_42)
total_tokens_42 = len(tokenizer_bert.encode(rag_string_42, add_special_tokens=True)) 
print(f"Q4 Answer: {total_tokens_42} tokens")

Q4 Answer: 216 tokens


### Question - 5

In [7]:
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"
res_q5 = zs(rag_string_150, candidate_labels=labels_150)
q5_prob = res_q5['scores'][res_q5['labels'].index(ans_150_text)]
print(f"Q5 Answer: {q5_prob:.3f}")

Q5 Answer: 0.989


### Question - 6

In [8]:
bad_doc_999 = kb[999]
adversarial_string = f"Context: {bad_doc_999} Question: {prompt_150}"
res_q6 = zs(adversarial_string, candidate_labels=labels_150)
q6_prob = res_q6['scores'][res_q6['labels'].index(ans_150_text)]
print(f"Q6 Answer: {q6_prob:.3f}")

Q6 Answer: 0.529


### Question - 7

In [9]:
hits = 0
for i in range(100):
    row_i = train.iloc[i]
    p_i = str(row_i['prompt'])
    ans_text_i = str(row_i[row_i['answer']])
    
    p_embed = model.encode([p_i], show_progress_bar=False)
    _, I_i = index.search(p_embed, k=5)
    retrieved_docs = [kb[idx] for idx in I_i[0]]
    
    if any(ans_text_i in doc for doc in retrieved_docs):
        hits += 1

hit_rate_pct = (hits / 100) * 100
print(f"Q7 Answer: {hit_rate_pct:.1f}%")

Q7 Answer: 73.0%


### Question - 8

In [10]:
def calculate_map3(actual_letter, predicted_labels_ordered, row_ref):
    # Map the text string labels back to choice keys ('A', 'B', 'C', 'D', 'E')
    mapping = {str(row_ref['A']): 'A', str(row_ref['B']): 'B', str(row_ref['C']): 'C', str(row_ref['D']): 'D', str(row_ref['E']): 'E'}
    predicted_letters = [mapping[label] for label in predicted_labels_ordered[:3]]
    
    # Calculating Mean Average Precision at 3
    score = 0.0
    for rank, letter in enumerate(predicted_letters):
        if letter == actual_letter:
            score = 1.0 / (rank + 1)
            break
    return score

map3_scores = []

for i in range(20):
    row_i = train.iloc[i]
    p_i = str(row_i['prompt'])
    labels_i = [str(row_i['A']), str(row_i['B']), str(row_i['C']), str(row_i['D']), str(row_i['E'])]
    true_letter = row_i['answer']
    
    #  Retrieve k=5
    p_embed = model.encode([p_i], show_progress_bar=False)
    _, I_i = index.search(p_embed, k=5)
    docs_5 = [kb[idx] for idx in I_i[0]]
    
    #  Rerank using CrossEncoder
    pairs_i = [[p_i, doc] for doc in docs_5]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_5[np.argmax(ce_scores_i)]
    
    #  Augment
    rag_string_i = f"Context: {best_doc} Question: {p_i}"
    
    #  Predict
    res_i = zs(rag_string_i, candidate_labels=labels_i)
    
    # Score MAP@3
    ordered_preds = res_i['labels'] 
    map3_val = calculate_map3(true_letter, ordered_preds, row_i)
    map3_scores.append(map3_val)

avg_map3 = np.mean(map3_scores)
print(f"Q8 Answer: {avg_map3:.3f}")

Q8 Answer: 0.975
